### What is the ISC Catalog?
• Global, authoritative source of earthquake and seismic phase data  
• Maintained by the International Seismological Centre (ISC)  
• Website: [https://www.isc.ac.uk/iscbulletin/search/catalogue/](https://www.isc.ac.uk/iscbulletin/search/catalogue/)  

### Why Use It?
• Long-term earthquake records  
• Reliable for global seismicity studies


### Seismic Event Extraction from ISC Catalog using ObsPy
# ======================================================

**Objective**: This notebook demonstrates how to use the `obspy` library to fetch seismic event data
from the ISC catalog for a specific geographic region and time range. The extracted data will include
the origin time, latitude, longitude, magnitude, and depth of each event. Finally, the data will be
saved to a CSV file.

---

### Importing Necessary Libraries
# -----------------------------

We'll begin by importing the necessary libraries. The `obspy` library will be used to interact with the
seismic data, and `pandas` will be used for data manipulation and saving to a CSV file.

---



In [1]:
import pandas as pd
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

ModuleNotFoundError: No module named 'obspy'

# Setting Up the FDSN Client
# --------------------------

The FDSN client allows us to access seismic data from various data centers.
In this tutorial, we will use the IRIS client to fetch events from the ISC catalog.

### Initialize the IRIS client
We use the `Client` class from the `obspy.clients.fdsn` module to connect to IRIS.

In [ ]:
client = Client("IRIS")

# Defining the Search Parameters
# ------------------------------

We'll define the geographic region, time range, and magnitude range for the events we want to extract.  
The region is defined by the latitude and longitude boundaries, and the time range is specified using the UTCDateTime format.


In [ ]:
# Define the range of catalog
minlat = 29
maxlat = 32
minlon = 77
maxlon = 82
starttime = UTCDateTime("2010-01-01")
endtime = UTCDateTime("2020-01-01")
minmag = 0
maxmag = 10

# Fetching Events from the ISC Catalog
# ------------------------------------


Using the defined parameters, we'll fetch the seismic event data from the ISC catalog.
The data includes events that occurred within the specified region and time range.


In [ ]:
cat = client.get_events(starttime=starttime, endtime=endtime,
                        minlatitude=minlat, maxlatitude=maxlat,
                        minlongitude=minlon, maxlongitude=maxlon,
                        minmagnitude=minmag, maxmagnitude=maxmag,
                        mindepth=0, maxdepth=100,
                        catalog="ISC")

# Extracting Event Details
# ------------------------

Now, we'll loop through the fetched events to extract relevant information such as the origin time, latitude, longitude, magnitude, and depth.
This data will be stored in a list.

# Prepare a list to store event details

In [ ]:
event_data = []

# Loop through the events and extract relevant information

In [ ]:
for event in cat:
    origin = event.preferred_origin() or event.origins[0]
    magnitude = event.preferred_magnitude() or event.magnitudes[0]

    # Extract latitude, longitude, magnitude, depth, and origin time
    lat = origin.latitude
    lon = origin.longitude
    mag = magnitude.mag
    depth = origin.depth / 1000 if origin.depth is not None else None  # Convert depth to kilometers
    origin_time = origin.time  # UTCDateTime

    # Append the event data (origin_time, lat, lon, mag, depth) to the list
    event_data.append([origin_time, lat, lon, mag, depth])

# Creating a DataFrame and Saving to CSV
# --------------------------------------

The extracted event data is then converted into a Pandas DataFrame, which makes it easier to manipulate and save to a file.
We'll save the DataFrame to a CSV file for further analysis.

# Convert the list to a Pandas DataFrame

In [ ]:
df = pd.DataFrame(event_data, columns=["Origin Time", "Latitude", "Longitude", "Magnitude", "Depth"])

# Save the DataFrame to a CSV file

In [ ]:
df.to_csv("./seismic_events.csv", index=False)

# Viewing the Extracted Data
# --------------------------

Finally, let's take a quick look at the first few rows of the extracted data to ensure everything was captured correctly.

# Print the DataFrame

In [ ]:
print(df)

# Seismicity Map using PyGMT
# --------------------------

Next, we will visualize the seismic events on a map using PyGMT.

In [ ]:
import pygmt

# Read the CSV file

In [ ]:
df = pd.read_csv("./seismic_events.csv")

# Determine the min and max latitude and longitude

In [ ]:
min_lat = df["Latitude"].min() - 0.1  # Subtract 1 degree
max_lat = df["Latitude"].max() + 0.1  # Add 1 degree
min_lon = df["Longitude"].min() - 0.1  # Subtract 1 degree
max_lon = df["Longitude"].max() + 0.1  # Add 1 degree
region = [min_lon, max_lon, min_lat, max_lat]  # [west, east, south, north]

# Create a simple map

In [ ]:
fig = pygmt.Figure()
fig.basemap(region=region, projection="M6i", frame=["+tSeismicity Map for Uttarakhand Himalaya"])
fig.coast(shorelines=True, land="lightgray", water="lightblue", frame=True)

# Create a coastline map with calculated region

In [ ]:
pygmt.makecpt(cmap="cool", series=[df.Depth.min(), df.Depth.max()], continuous=True)

# Plot the earthquake events

In [ ]:
fig.plot(x=df.Longitude,y=df.Latitude,size=0.02*(2 **df.Magnitude),fill=df.Depth,cmap=True,style="cc",pen="black")
fig.colorbar(frame='af+lFocal Depth (km)')

# Add inset map

In [ ]:
with fig.inset(position="jBL+w4c/2.4c+o0.1c", box="+gwhite+p1p"):
    fig.coast(
        region=[69.7, 95, 19.6, 33],
        projection="M4c",
        land="gray",
        borders=[1],
        shorelines="1/thin",
        water="white",
    )
    rectangle = [[min_lon, min_lat, max_lon, max_lat]]
    fig.plot(data=rectangle, style="r+s", pen="1p,red")
fig.text(text="INDIA", x=77.35, y=29.1, font="15p,Helvetica-Bold,white")

# Show the map

In [ ]:
fig.show()

# Save the figure

In [ ]:
fig.savefig("Uttarakhand_Seismicity.png", dpi=480)

## Exercise: Plotting a Seismicity Map of Iceland

### Goal:
Use PyGMT to create a seismicity map of Iceland using earthquake data from a catalog.

### Instructions:
1. **Query an earthquake catalog** using Obspy (e.g., from IRIS or another data provider).
2. Use the following parameters to filter the catalog:

   - **Latitude range:** 62° to 67°
   - **Longitude range:** -24° to -16°
   - **Time range:** 2010-01-01 to 2020-01-01
   - **Magnitude range:** 0 to 10

3. **Plot the seismic events** on a map of Iceland using PyGMT:
   - Use appropriate markers (e.g., circles sized by magnitude).
   - Add coastlines, water, and land color.
   - Include a legend and a title.

In [ ]:
# Define the range of catalog
minlat = **
maxlat = **
minlon = **
maxlon = **
starttime = UTCDateTime("****-**-**")
endtime = UTCDateTime("****-**-**")
minmag = *
maxmag = **

In [ ]:
# Fetch earthquake events from the ISC catalog using given parameters
cat = client.get_events(starttime=starttime, endtime=endtime,
                        minlatitude=minlat, maxlatitude=maxlat,
                        minlongitude=minlon, maxlongitude=maxlon,
                        minmagnitude=minmag, maxmagnitude=maxmag,
                        mindepth=0, maxdepth=100,
                        catalog="ISC")

# Initialize a list to store event data
event_data = []

# Loop through each event in the catalog
for event in cat:
    origin = event.preferred_origin() or event.origins[0]  # Get preferred origin or fallback to first
    magnitude = event.preferred_magnitude() or event.magnitudes[0]  # Get preferred magnitude or fallback

    # Extract relevant information from the origin and magnitude
    lat = origin.latitude  # Latitude of the epicenter
    lon = origin.longitude  # Longitude of the epicenter
    mag = magnitude.mag  # Magnitude of the earthquake
    depth = origin.depth / 1000 if origin.depth is not None else None  # Convert depth from meters to kilometers
    origin_time = origin.time  # Origin time in UTC

    # Add extracted data to the event_data list
    event_data.append([origin_time, lat, lon, mag, depth])

# Convert the list of events to a Pandas DataFrame
df = pd.DataFrame(event_data, columns=["Origin Time", "Latitude", "Longitude", "Magnitude", "Depth"])

# Save the DataFrame as a CSV file
df.to_csv("./seismic_events_iceland.csv", index=False)

# Reload the CSV to ensure data integrity (optional)
df = pd.read_csv("./seismic_events_iceland.csv")

# Define the map bounds based on event coordinates, with padding
min_lat = df["Latitude"].min() - 0.1  # Slightly expand south boundary
max_lat = df["Latitude"].max() + 0.1  # Slightly expand north boundary
min_lon = df["Longitude"].min() - 0.1  # Slightly expand west boundary
max_lon = df["Longitude"].max() + 0.1  # Slightly expand east boundary
region = [min_lon, max_lon, min_lat, max_lat]  # Define region as [W, E, S, N]

# Create a PyGMT figure
fig = pygmt.Figure()

# Set up the map projection and title
fig.basemap(region=region, projection="M6i", frame=["+tSeismicity Map for Iceland"])

# Draw coastlines, land, and water features
fig.coast(shorelines=True, land="lightgray", water="lightblue", frame=True)

# Create a color palette for depth values (cool colormap)
pygmt.makecpt(cmap="cool", series=[df.Depth.min(), df.Depth.max()], continuous=True)

# Plot the earthquake events:
# - Position: Longitude, Latitude
# - Size: scaled by magnitude (2^M)
# - Fill color: depth
# - Style: filled circle (cc), with black outline
fig.plot(x=df.Longitude,
         y=df.Latitude,
         size=0.02 * (2 ** df.Magnitude),
         fill=df.Depth,
         cmap=True,
         style="cc",
         pen="black")

# Add a colorbar for depth
fig.colorbar(frame='af+lFocal Depth (km)')

# Display the figure in the notebook
fig.show()

# Save the figure as a high-resolution PNG
fig.savefig("Iceland_Seismicity.png", dpi=480)
